# 01 — Data Preprocessing
**Goal:** Load raw WISDM phone sensor data, merge accelerometer and gyroscope by timestamp,
resample to a uniform 20 Hz grid, cut sliding windows, and save a ready-to-train `.npz` file.

Activities kept: A=Walking (0), B=Jogging (1), C=Stairs (2), D=Sitting (3), E=Standing (4)

In [1]:
# ── Standard-library imports ──────────────────────────────────────────────────
import os          # for building file paths
import glob        # for finding data_*.txt files
import warnings    # to suppress noisy pandas warnings

# ── Third-party imports ───────────────────────────────────────────────────────
import numpy as np          # numerical arrays
import pandas as pd         # DataFrame and time-series helpers

# Suppress FutureWarnings that pandas 2.x sometimes emits for resample
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
# ── Path constants ────────────────────────────────────────────────────────────
# Root of the raw sensor data (phone only, per task specification)
BASE_DIR   = os.path.join('..', 'wisdm-dataset', 'raw', 'phone')
ACCEL_DIR  = os.path.join(BASE_DIR, 'accel')   # accelerometer folder
GYRO_DIR   = os.path.join(BASE_DIR, 'gyro')    # gyroscope folder

# Where the processed windows will be saved
OUTPUT_DIR = os.path.join('..', 'data')
os.makedirs(OUTPUT_DIR, exist_ok=True)   # create ../data/ if it does not exist

print('Accel dir:', os.path.abspath(ACCEL_DIR))
print('Gyro  dir:', os.path.abspath(GYRO_DIR))

Accel dir: C:\Users\shiva\Downloads\HAR2\wisdm-dataset\raw\phone\accel
Gyro  dir: C:\Users\shiva\Downloads\HAR2\wisdm-dataset\raw\phone\gyro


In [3]:
# ── Activity mapping ──────────────────────────────────────────────────────────
# Only activities A-E are used; all others are dropped during loading
ACTIVITY_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# ── Window parameters ─────────────────────────────────────────────────────────
WINDOW_SIZE = 100    # 100 samples × 50 ms/sample = 5 seconds per window
STEP_SIZE   = 50     # 50-sample hop → 50% overlap between consecutive windows

# ── Sensor merge tolerance ───────────────────────────────────────────────────
# merge_asof will only pair a gyro row with an accel row if their timestamps
# are within 50 000 000 nanoseconds = 50 ms of each other
MERGE_TOL_NS = 50_000_000   # 50 ms in nanoseconds

print('Activity map:', ACTIVITY_MAP)
print(f'Window: {WINDOW_SIZE} samples, step: {STEP_SIZE} samples')

Activity map: {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
Window: 100 samples, step: 50 samples


In [4]:
# ── Helper: parse one raw .txt file ──────────────────────────────────────────
def parse_sensor_file(filepath, sensor_prefix):
    """Read a single data_XXXX_*.txt file and return a clean DataFrame.

    Parameters
    ----------
    filepath      : path to the .txt file
    sensor_prefix : 'a' for accelerometer columns, 'g' for gyroscope columns

    Returns
    -------
    DataFrame with columns: subject, activity, timestamp, {prefix}x, {prefix}y, {prefix}z
    """
    rows = []   # will hold cleaned rows

    # Open the file and iterate line by line
    with open(filepath, 'r', errors='replace') as f:
        for line in f:
            # Strip surrounding whitespace and the trailing semicolon
            line = line.strip().rstrip(';')

            # Skip blank lines (common at file end)
            if not line:
                continue

            # Split on comma into exactly 6 parts
            parts = line.split(',')
            if len(parts) != 6:
                continue   # skip any malformed lines

            # Unpack the six fields
            subject, activity, timestamp, x, y, z = parts

            # Keep only the five target activities; skip everything else
            activity = activity.strip()
            if activity not in ACTIVITY_MAP:
                continue

            # Convert types: subject→int, timestamp→int, x/y/z→float
            try:
                rows.append({
                    'subject'  : int(subject.strip()),
                    'activity' : activity,
                    'timestamp': int(timestamp.strip()),
                    f'{sensor_prefix}x': float(x.strip()),
                    f'{sensor_prefix}y': float(y.strip()),
                    f'{sensor_prefix}z': float(z.strip()),
                })
            except ValueError:
                continue   # skip rows with un-parseable numbers

    # Build a DataFrame from the collected rows
    df = pd.DataFrame(rows)
    return df

In [5]:
# ── Load ALL accelerometer files ──────────────────────────────────────────────
# glob pattern ensures we only pick data_*.txt, skipping .DS_Store etc.
accel_files = sorted(glob.glob(os.path.join(ACCEL_DIR, 'data_*.txt')))
print(f'Found {len(accel_files)} accelerometer files')

# Parse each file and collect into a list of DataFrames
accel_dfs = [parse_sensor_file(fp, 'a') for fp in accel_files]

# Concatenate all subjects into a single accelerometer DataFrame
accel_all = pd.concat(accel_dfs, ignore_index=True)
print(f'Accel rows loaded: {len(accel_all):,}')
print(accel_all.head(3))

Found 51 accelerometer files


Accel rows loaded: 1,338,067
   subject activity        timestamp        ax         ay        az
0     1600        A  252207666810782 -0.364761   8.793503  1.055084
1     1600        A  252207717164786 -0.879730   9.768784  1.016998
2     1600        A  252207767518790  2.001495  11.109070  2.619156


In [6]:
# ── Load ALL gyroscope files ───────────────────────────────────────────────────
gyro_files = sorted(glob.glob(os.path.join(GYRO_DIR, 'data_*.txt')))
print(f'Found {len(gyro_files)} gyroscope files')

# Parse with 'g' prefix so columns become gx, gy, gz
gyro_dfs = [parse_sensor_file(fp, 'g') for fp in gyro_files]

# Concatenate all subjects into a single gyroscope DataFrame
gyro_all = pd.concat(gyro_dfs, ignore_index=True)
print(f'Gyro rows loaded: {len(gyro_all):,}')
print(gyro_all.head(3))

Found 51 gyroscope files


Gyro rows loaded: 1,006,749
   subject activity        timestamp        gx        gy        gz
0     1600        A  252207918580802 -0.853210  0.297226  0.890182
1     1600        A  252207968934806 -0.875137  0.015472  0.162231
2     1600        A  252208019288809 -0.720169  0.388489 -0.284012


In [7]:
# ── Merge accelerometer and gyroscope by timestamp ───────────────────────────
# Strategy: within each (subject, activity) pair, use merge_asof so each
# accel row is matched to the nearest gyro row within MERGE_TOL_NS = 50 ms.
# merge_asof requires both DataFrames to be sorted by the key column.

# Sort both DataFrames by (subject, activity, timestamp) before merging
accel_sorted = accel_all.sort_values(['subject', 'activity', 'timestamp']).reset_index(drop=True)
gyro_sorted  = gyro_all.sort_values(['subject', 'activity', 'timestamp']).reset_index(drop=True)

merged_parts = []   # will hold merged DataFrames, one per (subject, activity)

# Iterate over every unique (subject, activity) combination present in accel
for (subj, act), accel_grp in accel_sorted.groupby(['subject', 'activity']):

    # Extract the matching gyro slice for the same subject+activity
    gyro_grp = gyro_sorted[
        (gyro_sorted['subject'] == subj) &
        (gyro_sorted['activity'] == act)
    ].copy()

    # Skip if this subject/activity is absent in gyro (should not happen)
    if gyro_grp.empty:
        continue

    # merge_asof: for each accel row, find the nearest gyro row by timestamp
    # direction='nearest' means it can look forward OR backward
    merged = pd.merge_asof(
        accel_grp.reset_index(drop=True),
        gyro_grp[['timestamp', 'gx', 'gy', 'gz']],
        on='timestamp',
        tolerance=MERGE_TOL_NS,
        direction='nearest'
    )

    # Drop rows where no gyro match was found within the tolerance window
    merged.dropna(subset=['gx', 'gy', 'gz'], inplace=True)

    merged_parts.append(merged)

# Combine all (subject, activity) merged blocks
merged_all = pd.concat(merged_parts, ignore_index=True)
print(f'Rows after merge: {len(merged_all):,}')
print(merged_all.columns.tolist())
print(merged_all.head(3))

Rows after merge: 1,336,403
['subject', 'activity', 'timestamp', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']
   subject activity        timestamp        ax         ay        az        gx  \
0     1600        A  252207918580802 -4.332779  13.361191 -0.718872 -0.853210   
1     1600        A  252207968934806 -0.319443  13.318359 -0.232025 -0.875137   
2     1600        A  252208019288809  1.566452   9.515274 -0.017776 -0.720169   

         gy        gz  
0  0.297226  0.890182  
1  0.015472  0.162231  
2  0.388489 -0.284012  


In [8]:
# ── Resample each (subject, activity) block to uniform 20 Hz ────────────────
# The raw timestamps are unevenly spaced and sometimes duplicated.
# We:
#   1. Convert the nanosecond integer timestamp to a proper DatetimeIndex.
#   2. Average any duplicates within the same millisecond bucket.
#   3. Resample to a 50 ms (= 20 Hz) regular grid using mean() to bin samples.
#   4. Linearly interpolate any remaining NaN gaps.
#   5. Drop leading/trailing NaN rows that interpolation cannot fill.

SENSOR_COLS = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']   # the six channel names

# Rename accel columns to match the target channel names
merged_all.rename(columns={'ax': 'ax', 'ay': 'ay', 'az': 'az'}, inplace=True)

resampled_parts = []   # one resampled DataFrame per (subject, activity)

for (subj, act), grp in merged_all.groupby(['subject', 'activity']):

    # Convert nanosecond timestamps to a pandas DatetimeIndex
    dt_index = pd.to_datetime(grp['timestamp'], unit='ns')

    # Build a DataFrame of the six sensor channels indexed by datetime
    sensor_df = grp[['ax', 'ay', 'az', 'gx', 'gy', 'gz']].copy()
    sensor_df.index = dt_index

    # Sort by time (merge_asof already sorted, but be safe)
    sensor_df.sort_index(inplace=True)

    # Resample to 50 ms grid: aggregate duplicate timestamps with mean
    resampled = sensor_df.resample('50ms').mean()

    # Linearly interpolate gaps (at most the number of missing 50 ms slots)
    resampled = resampled.interpolate(method='linear')

    # Drop any leading/trailing NaN rows that interpolation cannot fill
    resampled.dropna(inplace=True)

    # Attach subject and activity back as plain columns for later grouping
    resampled['subject']  = subj
    resampled['activity'] = act

    resampled_parts.append(resampled)

# Combine all resampled blocks into one DataFrame
resampled_all = pd.concat(resampled_parts)
print(f'Rows after resampling: {len(resampled_all):,}')
print(resampled_all[SENSOR_COLS].describe().round(3))

Rows after resampling: 906,306


               ax          ay          az          gx          gy          gz
count  906306.000  906306.000  906306.000  906306.000  906306.000  906306.000
mean        0.306      -2.136      -0.097      -0.005      -0.007      -0.005
std         6.038       8.379       4.598       1.254       1.452       1.018
min       -56.984     -71.158     -50.650     -24.771     -34.978     -34.985
25%        -3.070      -8.753      -2.332      -0.262      -0.269      -0.180
50%         0.312      -2.837      -0.387      -0.000      -0.000      -0.001
75%         3.651       3.407       2.079       0.274       0.260       0.184
max        72.566      39.840      37.027      23.529      31.623      13.166


In [9]:
# ── Sliding-window extraction ─────────────────────────────────────────────────
# For each (subject, activity) block, slice overlapping windows of WINDOW_SIZE
# rows with a stride of STEP_SIZE rows.
# A window NEVER mixes subjects or activities because we iterate within groups.

X_list        = []   # will collect window arrays, each shape (100, 6)
y_list        = []   # integer label for each window
subjects_list = []   # subject ID for each window (needed for split)

for (subj, act), grp in resampled_all.groupby(['subject', 'activity']):

    # Extract only the six sensor columns as a numpy array
    data = grp[SENSOR_COLS].values   # shape: (T, 6)

    # Convert activity letter to integer label using the activity map
    label = ACTIVITY_MAP[act]

    # Slide a window of WINDOW_SIZE across the block with STEP_SIZE hop
    for start in range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE           # exclusive end index
        window = data[start:end]            # shape: (100, 6)

        # Sanity check: ensure the window is exactly the right size
        if window.shape[0] == WINDOW_SIZE:
            X_list.append(window)
            y_list.append(label)
            subjects_list.append(subj)

# Convert lists to NumPy arrays
X        = np.array(X_list,        dtype=np.float32)  # shape: (N, 100, 6)
y        = np.array(y_list,        dtype=np.int32)     # shape: (N,)
subjects = np.array(subjects_list, dtype=np.int32)     # shape: (N,)

print(f'X shape:        {X.shape}   → (windows, timesteps, channels)')
print(f'y shape:        {y.shape}')
print(f'subjects shape: {subjects.shape}')

X shape:        (17685, 100, 6)   → (windows, timesteps, channels)
y shape:        (17685,)
subjects shape: (17685,)


In [10]:
# ── Print windows per class ───────────────────────────────────────────────────
# Reverse the activity map so we can look up the letter from the integer label
LABEL_TO_NAME = {v: k for k, v in ACTIVITY_MAP.items()}
ACTIVITY_NAMES = {
    'A': 'Walking', 'B': 'Jogging', 'C': 'Stairs',
    'D': 'Sitting', 'E': 'Standing'
}

print('\nWindows per class:')
print(f'{"Label":<6} {"Code":<6} {"Activity":<12} {"Windows":>8}')
print('-' * 36)
for label in sorted(ACTIVITY_MAP.values()):
    code  = LABEL_TO_NAME[label]
    name  = ACTIVITY_NAMES[code]
    count = int((y == label).sum())
    print(f'{label:<6} {code:<6} {name:<12} {count:>8}')
print(f'{"":<6} {"":<6} {"TOTAL":<12} {len(y):>8}')


Windows per class:
Label  Code   Activity      Windows
------------------------------------
0      A      Walking          3579
1      B      Jogging          3438
2      C      Stairs           3508
3      D      Sitting          3580
4      E      Standing         3580
              TOTAL           17685


In [11]:
# ── Save windows to disk ──────────────────────────────────────────────────────
# np.savez_compressed packs multiple arrays into one .npz file
output_path = os.path.join(OUTPUT_DIR, 'windows.npz')
np.savez_compressed(
    output_path,
    X=X,               # sensor data  → shape (N, 100, 6)
    y=y,               # integer labels→ shape (N,)
    subjects=subjects  # subject IDs  → shape (N,)
)

# Report the file size
size_mb = os.path.getsize(output_path) / 1_048_576
print(f'Saved to: {os.path.abspath(output_path)}')
print(f'File size: {size_mb:.1f} MB')
print(f'\nFinal shapes  →  X: {X.shape}, y: {y.shape}, subjects: {subjects.shape}')

Saved to: C:\Users\shiva\Downloads\HAR2\data\windows.npz
File size: 16.4 MB

Final shapes  →  X: (17685, 100, 6), y: (17685,), subjects: (17685,)


In [12]:
# ── Quick sanity check: reload and verify ─────────────────────────────────────
# Load the saved file and confirm the arrays match what we expect
loaded = np.load(output_path)
print('Keys in .npz file:', list(loaded.keys()))
print('X loaded shape:', loaded['X'].shape)
print('y loaded shape:', loaded['y'].shape)
print('subjects loaded shape:', loaded['subjects'].shape)

# Confirm data type and value range look reasonable
print(f'X dtype: {loaded["X"].dtype},  min={loaded["X"].min():.3f},  max={loaded["X"].max():.3f}')
print('Unique subject IDs:', np.unique(loaded['subjects']).tolist())
print('Unique labels     :', np.unique(loaded['y']).tolist())

Keys in .npz file: ['X', 'y', 'subjects']
X loaded shape: (17685, 100, 6)
y loaded shape: (17685,)
subjects loaded shape: (17685,)


X dtype: float32,  min=-71.158,  max=72.566
Unique subject IDs: [1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612, 1613, 1614, 1615, 1616, 1617, 1618, 1619, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1627, 1628, 1629, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1649, 1650]
Unique labels     : [0, 1, 2, 3, 4]
